In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import json

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

Log loaded. Rows: 9
PROCESSED_DIR: /Users/boulanger/Documents/governance-framework/data/processed


In [2]:
sources = [
    {
        "source_id": "VDEM",
        "source_name": "Varieties of Democracy (V-Dem)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv on full dataset download",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1789-present",
        "highest_tier": "P1",
        "notes": "Single largest source in framework. ~4000 variables. Download full dataset once; filter to needed variables. Codebook study essential before metric selection."
    },
    {
        "source_id": "WGI",
        "source_name": "World Bank Worldwide Governance Indicators",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "annual",
        "coverage_countries": 215,
        "coverage_years": "1996-present",
        "highest_tier": "P1",
        "notes": "Used as concept primary in 3 concepts (GE, PS, RQ); category cross-check elsewhere. wbgapi is clean and well-documented."
    },
    {
        "source_id": "WDI",
        "source_name": "World Bank World Development Indicators",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "annual",
        "coverage_countries": 217,
        "coverage_years": "1960-present",
        "highest_tier": "P1",
        "notes": "Used for service delivery sector indicators. Select specific indicators only — dataset is very broad."
    },
    {
        "source_id": "WJP",
        "source_name": "World Justice Project Rule of Law Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 142,
        "coverage_years": "2012-present",
        "highest_tier": "P1",
        "notes": "Borderline coverage (~142 countries). Factors allocated by concept: F2=Corruption, F3=Legal Quality+Transparency, F4=Legal Quality, F5=Personal Security+Stability, F6=Regulatory Quality, F7+F8=Judicial Independence. Download Excel from worldjusticeproject.org."
    },
    {
        "source_id": "FH_FIW",
        "source_name": "Freedom House Freedom in the World",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 210,
        "coverage_years": "1973-present",
        "highest_tier": "P1",
        "notes": "Disciplined sub-component extraction required. Sub-categories by concept: A=Electoral Process, D=Expression+Belief (Civil Liberties+Media), E=Associational Rights (Civil Society), G=Personal Autonomy (Civil Liberties). Do NOT use composite FIW score."
    },
    {
        "source_id": "FSI",
        "source_name": "Fragile States Index (Fund for Peace)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 179,
        "coverage_years": "2006-present",
        "highest_tier": "P1",
        "notes": "Different indicators by concept: P1 (Factionalized Elites) + S1 (Group Grievance) in Political Settlement; P2 (Public Services) in Service Delivery; C1 (Security Apparatus) in State Capacity."
    },
]

# Preview
pd.DataFrame(sources)

,source_id,source_name,access_method,python_approach,update_frequency,coverage_countries,coverage_years,highest_tier,notes
0,VDEM,Varieties of Democracy (V-Dem),bulk_download,pd.read_csv on full dataset download,annual,180,1789-present,P1,Single largest source in framework. ~4000 vari...
1,WGI,World Bank Worldwide Governance Indicators,api,wbgapi,annual,215,1996-present,P1,"Used as concept primary in 3 concepts (GE, PS,..."
2,WDI,World Bank World Development Indicators,api,wbgapi,annual,217,1960-present,P1,Used for service delivery sector indicators. S...
3,WJP,World Justice Project Rule of Law Index,bulk_download,pd.read_excel,annual,142,2012-present,P1,Borderline coverage (~142 countries). Factors ...
4,FH_FIW,Freedom House Freedom in the World,bulk_download,pd.read_excel,annual,210,1973-present,P1,Disciplined sub-component extraction required....
5,FSI,Fragile States Index (Fund for Peace),bulk_download,pd.read_excel,annual,179,2006-present,P1,Different indicators by concept: P1 (Factional...


In [3]:
sources += [
    {
        "source_id": "IMF_FISCAL_RULES",
        "source_name": "IMF Fiscal Rules Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 100,
        "coverage_years": "1985-present",
        "highest_tier": "P1",
        "notes": "Used in Macroeconomic policy framework. Covers existence, design, and compliance of fiscal rules. Download from IMF website."
    },
    {
        "source_id": "IMF_AREAER",
        "source_name": "IMF Annual Report on Exchange Arrangements and Exchange Restrictions",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 190,
        "coverage_years": "1950-present",
        "highest_tier": "P1",
        "notes": "De jure exchange rate regime classification. Used in Macroeconomic policy framework alongside Reinhart-Rogoff de facto classifications."
    },
    {
        "source_id": "IMF_IMAPP",
        "source_name": "IMF Integrated Macroprudential Policy Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 130,
        "coverage_years": "1990-present",
        "highest_tier": "P1",
        "notes": "Macroprudential policy adoption. Used in Macroeconomic policy framework. Download from IMF website."
    },
    {
        "source_id": "IMF_SPI",
        "source_name": "World Bank Statistical Performance Indicators",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "annual",
        "coverage_countries": 174,
        "coverage_years": "2016-present",
        "highest_tier": "P1",
        "notes": "Primary source for Statistical and informational infrastructure. Covers data infrastructure, sources, products, services, use."
    },
    {
        "source_id": "ROMELLI_CBI",
        "source_name": "Romelli Central Bank Independence Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 155,
        "coverage_years": "1923-2021",
        "highest_tier": "P1",
        "notes": "Current state-of-the-art for CBI. Updates require new paper/replication release — not annually updated. Verify currency at metric pass."
    },
    {
        "source_id": "DINCER_CB",
        "source_name": "Dincer-Eichengreen Central Bank Transparency Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 120,
        "coverage_years": "1998-present",
        "highest_tier": "P1",
        "notes": "CB transparency dimension for Macroeconomic policy framework. Update status and most recent year to verify at metric pass."
    },
    {
        "source_id": "REINHART_ROGOFF",
        "source_name": "Reinhart-Rogoff Exchange Rate Classifications",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 190,
        "coverage_years": "1940-present",
        "highest_tier": "P1",
        "notes": "De facto exchange rate regime. Complements AREAER de jure. Update frequency irregular — verify most recent release."
    },
    {
        "source_id": "UNODC_HOMICIDE",
        "source_name": "UNODC Homicide Statistics",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "1990-present",
        "highest_tier": "P1",
        "notes": "Gold standard for Personal security and order. High S/N. Download from UNODC data portal."
    },
    {
        "source_id": "PTS",
        "source_name": "Political Terror Scale",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 190,
        "coverage_years": "1976-present",
        "highest_tier": "P1",
        "notes": "Used in both Personal security (P1) and Civil liberties (P2). Indicator repetition tracked. Download from politicalterrorscale.org."
    },
    {
        "source_id": "POWELL_THYNE",
        "source_name": "Powell-Thyne Coup Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "1950-present",
        "highest_tier": "P1",
        "notes": "Event-level coup data. Used in Political stability (P1) and Political settlement (Sp). Requires country-year panel construction from event data."
    },
    {
        "source_id": "UCDP",
        "source_name": "Uppsala Conflict Data Program",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "1946-present",
        "highest_tier": "P1",
        "notes": "Gold standard for armed conflict. Used in Political stability. Multiple datasets (GED, PRIO-Grid, dyadic). Use GED for country-year panel."
    },
    {
        "source_id": "ACLED",
        "source_name": "Armed Conflict Location and Event Data",
        "access_method": "api",
        "python_approach": "requests (ACLED API requires registration)",
        "update_frequency": "continuous",
        "coverage_countries": 250,
        "coverage_years": "1997-present",
        "highest_tier": "P1",
        "notes": "Real-time event data. API requires free registration and key. Complements UCDP with broader event types."
    },
    {
        "source_id": "TI_CPI",
        "source_name": "Transparency International Corruption Perceptions Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1995-present",
        "highest_tier": "P1",
        "notes": "Standard corruption measure. Used in Control of corruption (P1). Aggregator of 13 underlying sources."
    },
    {
        "source_id": "RSF_WPFI",
        "source_name": "RSF World Press Freedom Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "2002-present",
        "highest_tier": "P1",
        "notes": "Primary for Media freedom. Download from RSF website. Methodology changed significantly in 2023 — treat pre/post 2023 as partially discontinuous series."
    },
    {
        "source_id": "CPJ",
        "source_name": "Committee to Protect Journalists",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv or requests",
        "update_frequency": "continuous",
        "coverage_countries": 200,
        "coverage_years": "1992-present",
        "highest_tier": "P1",
        "notes": "Journalist safety outcome measure for Media freedom. Event-level data requiring country-year aggregation. CPJ database accessible via website download."
    },
    {
        "source_id": "CIVICUS",
        "source_name": "CIVICUS Monitor",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel or requests",
        "update_frequency": "annual",
        "coverage_countries": 197,
        "coverage_years": "2017-present",
        "highest_tier": "P1",
        "notes": "Used in Political participation (P1) and Civil society space (P1). Categorical scoring (Open/Narrowed/Obstructed/Repressed/Closed). Short time series."
    },
    {
        "source_id": "IDEA_EMB",
        "source_name": "IDEA Electoral Management Design Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 220,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "EMB design and independence for Electoral process. Structural/design data rather than time series. Update frequency to verify."
    },
    {
        "source_id": "PEI",
        "source_name": "Electoral Integrity Project — Perceptions of Electoral Integrity",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "per_election",
        "coverage_countries": 170,
        "coverage_years": "2012-present",
        "highest_tier": "P1",
        "notes": "Per-election cadence requires constructing most-recent-election panel. 49 indicators across 11 dimensions. Download from Electoral Integrity Project website."
    },
    {
        "source_id": "CCP",
        "source_name": "Comparative Constitutions Project",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 200,
        "coverage_years": "1789-present",
        "highest_tier": "P1",
        "notes": "De jure constitutional framework. Used across Legal quality, Judicial independence, Property rights, Legislative checks, Electoral process. Constitute Project is the searchable interface."
    },
    {
        "source_id": "DPI",
        "source_name": "Database of Political Institutions",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1975-present",
        "highest_tier": "P2",
        "notes": "Party fragmentation and government composition proxies. Used in Political settlement (P2). World Bank hosted."
    },
    {
        "source_id": "GPI",
        "source_name": "Global Peace Index (IEP)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 163,
        "coverage_years": "2008-present",
        "highest_tier": "P2",
        "notes": "Used in Political stability (P2) and Personal security (P2). Pre-aggregated composite — use domain-level scores not headline."
    },
    {
        "source_id": "ODIN",
        "source_name": "Open Data Inventory (Open Data Watch)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "biennial",
        "coverage_countries": 195,
        "coverage_years": "2016-present",
        "highest_tier": "P1",
        "notes": "Best-in-class for accessibility/openness sub-dimension of Statistical infrastructure. Also supplementary in Government transparency."
    },
    {
        "source_id": "PEFA",
        "source_name": "Public Expenditure and Financial Accountability",
        "access_method": "tier3_pdf",
        "python_approach": "PDF extraction — pdfplumber + manual review",
        "update_frequency": "per_country_4_7yr",
        "coverage_countries": 150,
        "coverage_years": "2001-present",
        "highest_tier": "P1",
        "notes": "Gold standard for PFM. Irregular per-country timing is the key operational challenge. PEFA Secretariat portal has structured data for some indicators. Verify portal coverage before committing to PDF extraction."
    },
    {
        "source_id": "OBS",
        "source_name": "Open Budget Survey (IBP)",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "biennial",
        "coverage_countries": 120,
        "coverage_years": "2006-present",
        "highest_tier": "P1",
        "notes": "Used in PFM (P1) and Government transparency (P2). Biennial cadence. Download from IBP website."
    },
    {
        "source_id": "FATF",
        "source_name": "FATF Mutual Evaluation Ratings",
        "access_method": "tier2_structured",
        "python_approach": "requests or pd.read_html from fatf-gafi.org",
        "update_frequency": "per_country_10yr",
        "coverage_countries": 200,
        "coverage_years": "2004-present",
        "highest_tier": "P1",
        "notes": "AML/CFT compliance ratings. Structured ratings on fatf-gafi.org are scrapeable. Full reports are PDFs (Tier 3). Per-country cycle ~10 years with intermediate follow-ups."
    },
    {
        "source_id": "BASEL_AML",
        "source_name": "Basel AML Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 205,
        "coverage_years": "2012-present",
        "highest_tier": "P1",
        "notes": "AML/CFT risk composite synthesising FATF and other sources. Annual, free, broad coverage. Practical workhorse for financial sector regulatory concept."
    },
    {
        "source_id": "HERITAGE_TR",
        "source_name": "Heritage Index of Economic Freedom — Trade Freedom",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1995-present",
        "highest_tier": "P1",
        "notes": "Used in Trade governance. Lower ideological loading for trade openness dimension than other Heritage components."
    },
    {
        "source_id": "HERITAGE_PR",
        "source_name": "Heritage Index of Economic Freedom — Property Rights",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 180,
        "coverage_years": "1995-present",
        "highest_tier": "P1",
        "notes": "Used in Property rights and contract enforcement. Lower loading than other Heritage components for this dimension."
    },
    {
        "source_id": "WB_LPI",
        "source_name": "World Bank Logistics Performance Index",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "irregular",
        "coverage_countries": 139,
        "coverage_years": "2007-present",
        "highest_tier": "P1",
        "notes": "Trade administration quality. 5-year update gap historically. Verify current cadence. Available via WB API."
    },
    {
        "source_id": "OECD_TFI",
        "source_name": "OECD Trade Facilitation Indicators",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "biennial",
        "coverage_countries": 163,
        "coverage_years": "2012-present",
        "highest_tier": "P1",
        "notes": "11 indicators covering trade administration. Updated every 2-3 years. Download from OECD website."
    },
    {
        "source_id": "KOF_TRADE",
        "source_name": "KOF Globalisation Index — Trade subindex",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "1970-present",
        "highest_tier": "P1",
        "notes": "De jure and de facto trade openness. Lower ideological framing than Heritage/Fraser. Download from KOF Swiss Economic Institute."
    },
    {
        "source_id": "UNCTAD_NTM",
        "source_name": "UNCTAD Non-Tariff Measures Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 110,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Only good cross-country NTM source. Borderline coverage (~110). Update frequency irregular. Download from UNCTAD website."
    },
    {
        "source_id": "YALE_EPI",
        "source_name": "Yale Environmental Performance Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "biennial",
        "coverage_countries": 180,
        "coverage_years": "2006-present",
        "highest_tier": "P1",
        "notes": "Used in Environmental governance. Use policy/institutional sub-components only — not headline composite. Biennial."
    },
    {
        "source_id": "CLIMATE_LAWS",
        "source_name": "LSE Grantham Climate Laws Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv or requests",
        "update_frequency": "continuous",
        "coverage_countries": 200,
        "coverage_years": "1800-present",
        "highest_tier": "P1",
        "notes": "De jure environmental and climate framework. Continuously updated. Download from climatecasechart.com / climate-laws.org."
    },
    {
        "source_id": "ND_GAIN",
        "source_name": "ND-GAIN Country Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 192,
        "coverage_years": "1995-present",
        "highest_tier": "P1",
        "notes": "Governance and readiness sub-scores for Environmental governance. Use sub-scores not headline index. Download from gain.nd.edu."
    },
    {
        "source_id": "IRENA_CAPACITY",
        "source_name": "IRENA Renewables Capacity Statistics",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "2000-present",
        "highest_tier": "P1",
        "notes": "Renewables outcomes for Environmental governance. High S/N for renewables specifically. Download from IRENA website."
    },
    {
        "source_id": "IRENA_POLICY",
        "source_name": "IRENA Renewable Energy Policies Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel or requests",
        "update_frequency": "continuous",
        "coverage_countries": 200,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Renewables policy adoption. Complements IRENA capacity outcomes. Download from IRENA website."
    },
    {
        "source_id": "WB_CARBON",
        "source_name": "World Bank Carbon Pricing Dashboard",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel or requests",
        "update_frequency": "annual",
        "coverage_countries": 100,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Carbon pricing existence and design. Universal coverage for countries with carbon pricing. Download from World Bank."
    },
    {
        "source_id": "HANSON_SIGMAN",
        "source_name": "Hanson-Sigman State Capacity Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 169,
        "coverage_years": "1960-2021",
        "highest_tier": "Sp",
        "notes": "Supplementary cross-check for State capacity. Last update 2021. Double-counting caveat — incorporates V-Dem and other framework sources."
    },
    {
        "source_id": "WTO_TFA",
        "source_name": "WTO Trade Facilitation Agreement Implementation",
        "access_method": "bulk_download",
        "python_approach": "requests or pd.read_html",
        "update_frequency": "continuous",
        "coverage_countries": 164,
        "coverage_years": "2017-present",
        "highest_tier": "P1",
        "notes": "Country commitments and implementation. All WTO members. Continuously updated on WTO website."
    },
    {
        "source_id": "IPU_PARLINE",
        "source_name": "IPU Parline Database",
        "access_method": "bulk_download",
        "python_approach": "requests or pd.read_html",
        "update_frequency": "continuous",
        "coverage_countries": 190,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Legislative structural features and oversight powers. Universal coverage. Authoritative source. Continuously updated."
    },
    {
        "source_id": "RTI_RATING",
        "source_name": "Centre for Law and Democracy / Access Info Europe RTI Rating",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel or requests",
        "update_frequency": "irregular",
        "coverage_countries": 138,
        "coverage_years": "2011-present",
        "highest_tier": "P2",
        "notes": "FOI/RTI legislation quality. Borderline coverage (~138). Used in Media freedom (P2) and Government transparency (P1)."
    },
    {
        "source_id": "TI_POLFINANCE",
        "source_name": "Transparency International Political Finance Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 180,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "Unique to Government transparency concept. Political party and campaign finance transparency."
    },
    {
        "source_id": "WIPO",
        "source_name": "WIPO IP Statistics",
        "access_method": "api",
        "python_approach": "requests (WIPO API)",
        "update_frequency": "annual",
        "coverage_countries": 190,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "IP protection dimension for Property rights. Download from WIPO IP Statistics portal."
    },
    {
        "source_id": "ILO_SOCIAL",
        "source_name": "ILO Social Security Coverage",
        "access_method": "api",
        "python_approach": "requests (ILO API) or pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 150,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "Formality via state administrative systems. Proxy for state reach for State capacity concept."
    },
    {
        "source_id": "WB_INFORMAL",
        "source_name": "World Bank Informal Economy Database",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 160,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "Informality as proxy for state reach. Used in State capacity (P2). Update frequency to verify."
    },
    {
        "source_id": "FRASER_REG",
        "source_name": "Fraser Economic Freedom — Regulation area",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 165,
        "coverage_years": "1970-present",
        "highest_tier": "P2",
        "notes": "Used in Regulatory quality (P2) with framing caveats. Download from Fraser Institute."
    },
    {
        "source_id": "FRASER_LEGAL",
        "source_name": "Fraser Economic Freedom — Legal System and Property Rights",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 165,
        "coverage_years": "1970-present",
        "highest_tier": "P2",
        "notes": "Used selectively in Property rights (P2) — property sub-components only. Judicial independence content stays in dedicated concept."
    },
    {
        "source_id": "PEW_GRI",
        "source_name": "Pew Government Restrictions Index and Social Hostilities Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "2007-present",
        "highest_tier": "P2",
        "notes": "Religious freedom dimension for Civil liberties. Tier 2 — religious freedom less central to political accountability than expression/dissent."
    },
    {
        "source_id": "WB_WBL",
        "source_name": "World Bank Women, Business and the Law",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "annual",
        "coverage_countries": 190,
        "coverage_years": "1971-present",
        "highest_tier": "P2",
        "notes": "Gender equality dimension for Civil liberties. Direct legal protections measurement. High S/N."
    },
    {
        "source_id": "NELDA",
        "source_name": "National Elections Across Democracy and Autocracy",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 200,
        "coverage_years": "1945-present",
        "highest_tier": "P2",
        "notes": "Event-level election data for Electoral process (P2). Requires country-year construction. Update status to verify."
    },
    {
        "source_id": "IDEA_PARTIP",
        "source_name": "IDEA Global State of Democracy — Participatory Engagement",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "annual",
        "coverage_countries": 173,
        "coverage_years": "1975-present",
        "highest_tier": "P2",
        "notes": "Used in Political participation (P2). Some V-Dem double-counting given underlying sources. Download from IDEA website."
    },
    {
        "source_id": "BCI",
        "source_name": "Bayesian Corruption Indicator",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 190,
        "coverage_years": "varies",
        "highest_tier": "P2",
        "notes": "Latent-variable cross-check for Control of corruption. Update currency to verify at metric pass."
    },
    {
        "source_id": "GLOBAL_DATA_BAROMETER",
        "source_name": "Global Data Barometer",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 109,
        "coverage_years": "2021-present",
        "highest_tier": "Sp",
        "notes": "Open data dimension. Borderline coverage (~109). Unique to Government transparency. Successor to defunct Open Data Barometer."
    },
    {
        "source_id": "IMF_SPI_SDDS",
        "source_name": "IMF Data Standards Subscriptions (SDDS/eGDDS)",
        "access_method": "bulk_download",
        "python_approach": "requests or pd.read_html",
        "update_frequency": "continuous",
        "coverage_countries": 190,
        "coverage_years": "1996-present",
        "highest_tier": "P2",
        "notes": "De jure standards compliance for Statistical infrastructure. Universal IMF members. Tiered by income group."
    },
    {
        "source_id": "WHO_GHO",
        "source_name": "WHO Global Health Observatory",
        "access_method": "api",
        "python_approach": "requests (WHO GHO API)",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Health-specific direct outputs for Service delivery. Universal coverage. Well-documented API."
    },
    {
        "source_id": "UNESCO_UIS",
        "source_name": "UNESCO Institute for Statistics",
        "access_method": "api",
        "python_approach": "requests (UIS API) or bulk download",
        "update_frequency": "annual",
        "coverage_countries": 200,
        "coverage_years": "varies",
        "highest_tier": "P1",
        "notes": "Education-specific direct outputs for Service delivery. Universal coverage. UIS bulk data download also available."
    },
    {
        "source_id": "UNDP_HDI",
        "source_name": "UNDP Human Development Index sub-indicators",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "annual",
        "coverage_countries": 193,
        "coverage_years": "1990-present",
        "highest_tier": "P1",
        "notes": "Use sub-indicators only (life expectancy, schooling years) — NOT composite HDI. Download from UNDP HDR website."
    },
    {
        "source_id": "WB_HCI",
        "source_name": "World Bank Human Capital Index",
        "access_method": "api",
        "python_approach": "wbgapi",
        "update_frequency": "biennial",
        "coverage_countries": 170,
        "coverage_years": "2018-present",
        "highest_tier": "P2",
        "notes": "Composite cross-check for Service delivery. Short time series. Use as summary cross-check not primary."
    },
    {
        "source_id": "POLITY5",
        "source_name": "Polity5",
        "access_method": "bulk_download",
        "python_approach": "pd.read_excel",
        "update_frequency": "irregular",
        "coverage_countries": 167,
        "coverage_years": "1800-2018",
        "highest_tier": "P2",
        "notes": "Limited use given V-Dem supersession. Durable (Sp) in Political stability; XCONST (P2) in Legislative checks; electoral components (Sp) in Electoral process. Update reliability concern — verify currency."
    },
    {
        "source_id": "LINZER_STATON",
        "source_name": "Linzer-Staton Judicial Independence Index",
        "access_method": "bulk_download",
        "python_approach": "pd.read_csv",
        "update_frequency": "irregular",
        "coverage_countries": 200,
        "coverage_years": "1948-present",
        "highest_tier": "Sp",
        "notes": "Supplementary cross-check for Judicial independence. Methodologically sophisticated latent variable. Update frequency to verify."
    },
    {
        "source_id": "ICNL",
        "source_name": "ICNL Civic Freedom Monitor",
        "access_method": "tier3_web",
        "python_approach": "requests / manual review",
        "update_frequency": "irregular",
        "coverage_countries": 100,
        "coverage_years": "varies",
        "highest_tier": "Sp",
        "notes": "De jure legal framework for Civil society space. Uneven coverage. Supplementary only."
    },
]

# Full registry
registry_df = pd.DataFrame(sources)
print(f"Total sources: {len(registry_df)}")
# registry_df
# with pd.option_context('display.max_rows', None):
#     display(registry_df)
registry_df

Total sources: 68


,source_id,source_name,access_method,python_approach,update_frequency,coverage_countries,coverage_years,highest_tier,notes
0,VDEM,Varieties of Democracy (V-Dem),bulk_download,pd.read_csv on full dataset download,annual,180,1789-present,P1,Single largest source in framework. ~4000 vari...
1,WGI,World Bank Worldwide Governance Indicators,api,wbgapi,annual,215,1996-present,P1,"Used as concept primary in 3 concepts (GE, PS,..."
2,WDI,World Bank World Development Indicators,api,wbgapi,annual,217,1960-present,P1,Used for service delivery sector indicators. S...
3,WJP,World Justice Project Rule of Law Index,bulk_download,pd.read_excel,annual,142,2012-present,P1,Borderline coverage (~142 countries). Factors ...
4,FH_FIW,Freedom House Freedom in the World,bulk_download,pd.read_excel,annual,210,1973-present,P1,Disciplined sub-component extraction required....
...,...,...,...,...,...,...,...,...,...
63,UNDP_HDI,UNDP Human Development Index sub-indicators,bulk_download,pd.read_csv,annual,193,1990-present,P1,"Use sub-indicators only (life expectancy, scho..."
64,WB_HCI,World Bank Human Capital Index,api,wbgapi,biennial,170,2018-present,P2,Composite cross-check for Service delivery. Sh...
65,POLITY5,Polity5,bulk_download,pd.read_excel,irregular,167,1800-2018,P2,Limited use given V-Dem supersession. Durable ...
66,LINZER_STATON,Linzer-Staton Judicial Independence Index,bulk_download,pd.read_csv,irregular,200,1948-present,Sp,Supplementary cross-check for Judicial indepen...


In [4]:
import os

output_path = os.path.join(PROCESSED_DIR, "source_registry.csv")
registry_df.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {registry_df.shape}")

Written: /Users/boulanger/Documents/governance-framework/data/processed/source_registry.csv
Shape: (68, 9)


In [5]:
# Update WHO_GHO entry to note it's subsumed by WDI
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'notes'] = (
    "Health workforce and outcomes data sourced from WHO Global Health Workforce Statistics. "
    "All required indicators available via WDI (wbgapi). "
    "Standalone WHO GHO pipeline not built — GHO OData API deprecated end-2025. "
    "Indicators covered: physicians/1000, nurses/1000, hospital beds/1000, UHC coverage index."
)
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'python_approach'] = 'wbgapi — see WDI pipeline'

# Save updated registry
output_path = os.path.join(PROCESSED_DIR, "source_registry.csv")
registry_df.to_csv(output_path, index=False)
print("Registry updated")
registry_df[registry_df['source_id'] == 'WHO_GHO'][['source_id', 'access_method', 'python_approach', 'notes']]

Registry updated


,source_id,access_method,python_approach,notes
61,WHO_GHO,via_wdi,wbgapi — see WDI pipeline,Health workforce and outcomes data sourced fro...


In [6]:
# Update WHO_GHO entry to reflect it's subsumed by WDI
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'notes'] = (
    "Health workforce and outcomes data sourced from WHO Global Health Workforce Statistics. "
    "All required indicators available via WDI (wbgapi). "
    "Standalone WHO GHO pipeline not built — GHO OData API deprecated end-2025. "
    "Indicators covered: physicians/1000, nurses/1000, hospital beds/1000, UHC coverage index."
)
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WHO_GHO', 'python_approach'] = 'wbgapi — see WDI pipeline'

# Save
registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'WHO_GHO'][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
   source_id access_method            python_approach
61   WHO_GHO       via_wdi  wbgapi — see WDI pipeline


In [7]:
import json
from pathlib import Path

notebooks_dir = Path(PROJECT_ROOT) / 'notebooks' / 'exploration'
for nb_path in sorted(notebooks_dir.glob('*.ipynb')):
    with open(nb_path) as f:
        nb = json.load(f)
    first_cell = ''.join(nb['cells'][0]['source'])
    if 'PROCESSED_DIR' in first_cell and 'from config import' not in first_cell:
        print(f"⚠️  HARDCODED: {nb_path.name}")
    else:
        print(f"OK: {nb_path.name}")

OK: 01_pdf_extraction.ipynb
OK: 02_source_registry.ipynb
OK: 03_vdem_pipeline.ipynb
OK: 04_wgi_pipeline.ipynb
OK: 05_wjp_pipeline.ipynb
OK: 06_fh_fiw_pipeline.ipynb
OK: 07_fsi_pipeline.ipynb
OK: 08_ti_cpi_pipeline.ipynb
OK: 09_wdi_pipeline.ipynb
OK: 10_imf_spi_pipeline.ipynb
OK: 11_acled_pipeline.ipynb
OK: 12_ucdp_pipeline.ipynb


In [8]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'UNESCO_UIS', 'notes'] = (
    "Education indicators sourced from UNESCO UIS, distributed via WDI (wbgapi). "
    "Standalone UNESCO UIS pipeline not built. "
    "Indicators covered: education expenditure % GDP, education expenditure % govt, "
    "pupil-teacher ratios primary and secondary."
)
registry_df.loc[registry_df['source_id'] == 'UNESCO_UIS', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'UNESCO_UIS', 'python_approach'] = 'wbgapi — see WDI pipeline'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'UNESCO_UIS'][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
     source_id access_method            python_approach
62  UNESCO_UIS       via_wdi  wbgapi — see WDI pipeline


In [9]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

# WB_LPI
registry_df.loc[registry_df['source_id'] == 'WB_LPI', 'notes'] = (
    "Logistics Performance Index overall score. Available via WDI (LP.LPI.OVRL.XQ). "
    "Added to WDI pipeline. No standalone LPI pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'WB_LPI', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WB_LPI', 'python_approach'] = 'wbgapi — see WDI pipeline'

# WB_HCI
registry_df.loc[registry_df['source_id'] == 'WB_HCI', 'notes'] = (
    "Standard HCI (HD.HCI.OVRL) not available via WB API. "
    "Using HCI+ overall total (HD_HCIP_OVRL_TO) as substitute — same concept, expanded methodology. "
    "Added to WDI pipeline. No standalone HCI pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'WB_HCI', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WB_HCI', 'python_approach'] = 'wbgapi — see WDI pipeline'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'].isin(['WB_LPI', 'WB_HCI'])][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
   source_id access_method            python_approach
34    WB_LPI       via_wdi  wbgapi — see WDI pipeline
64    WB_HCI       via_wdi  wbgapi — see WDI pipeline


In [10]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

# WIPO
registry_df.loc[registry_df['source_id'] == 'WIPO', 'notes'] = (
    "Patent and trademark application data available via WDI (wbgapi). "
    "Indicators: IP.PAT.RESD, IP.PAT.NRES, IP.TMK.RSCT, IP.TMK.NRCT. "
    "All added to WDI pipeline. No standalone WIPO pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'WIPO', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'WIPO', 'python_approach'] = 'wbgapi — see WDI pipeline'

# ILO_SOCIAL
registry_df.loc[registry_df['source_id'] == 'ILO_SOCIAL', 'notes'] = (
    "Social protection coverage indicators available via World Bank API (wbgapi). "
    "Indicators: per_allsp.cov_pop_tot, per_sa_allsa.cov_pop_tot, per_si_allsi.cov_pop_tot. "
    "All added to WDI pipeline. No standalone ILO pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'ILO_SOCIAL', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'ILO_SOCIAL', 'python_approach'] = 'wbgapi — see WDI pipeline'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'].isin(['WIPO', 'ILO_SOCIAL'])][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
     source_id access_method            python_approach
49        WIPO       via_wdi  wbgapi — see WDI pipeline
50  ILO_SOCIAL       via_wdi  wbgapi — see WDI pipeline


In [11]:
registry_df = pd.read_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"))

registry_df.loc[registry_df['source_id'] == 'UNDP_HDI', 'notes'] = (
    "HDI sub-indicators available via WDI (wbgapi). "
    "Indicators added to WDI pipeline: SP.DYN.LE00.IN (life expectancy), NY.GNP.PCAP.PP.CD (GNI per capita PPP). "
    "Years of schooling not available via WDI — enrollment rates used as substitute. "
    "Composite HDI index not used per framework design. No standalone UNDP pipeline needed."
)
registry_df.loc[registry_df['source_id'] == 'UNDP_HDI', 'access_method'] = 'via_wdi'
registry_df.loc[registry_df['source_id'] == 'UNDP_HDI', 'python_approach'] = 'wbgapi — see WDI pipeline'

registry_df.to_csv(os.path.join(PROCESSED_DIR, "source_registry.csv"), index=False)
print("Registry updated")
print(registry_df[registry_df['source_id'] == 'UNDP_HDI'][['source_id', 'access_method', 'python_approach']].to_string())

Registry updated
   source_id access_method            python_approach
63  UNDP_HDI       via_wdi  wbgapi — see WDI pipeline
